In [2]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split

class InputDataset(Dataset):
  def __init__(self, df, tokenizer, max_len=128):
    self.encodings = tokenizer(
      list(df["text"]),
      truncation=True,
      padding=True,
      max_length=max_len,
      return_tensors="pt"
    )
    self.labels = list(df["label"])

  def __len__(self):
    return len(self.labels)

  def __getitem__(self, idx):
    return {
      "input_ids": self.encodings["input_ids"][idx],
      "attention_mask": self.encodings["attention_mask"][idx],
      "labels": torch.tensor(self.labels[idx], dtype=torch.long)
    }

# bert to rank the jokes/lyrics
def train_model(filename, labelname, output_dir, num_samples):
  df = pd.read_csv(filename)
  df = df.sample(n=num_samples, random_state=42)
  df["label"] = df[labelname].astype(int)

  train_df, eval_df = train_test_split(df, test_size=0.2, random_state=42)

  tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
  model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

  train_dataset = InputDataset(train_df, tokenizer)
  eval_dataset = InputDataset(eval_df,  tokenizer)

  training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_dir="./logs",
  )

  trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
  )

  trainer.train()
  model.save_pretrained(output_dir)
  tokenizer.save_pretrained(output_dir)

In [3]:
from transformers import pipeline

def classify_text(texts, model_dir):
  classifier = pipeline(
    "text-classification",
    model=model_dir,
    tokenizer=model_dir,
    return_all_scores=True
  )

  def score(text):
    output = classifier(text)
    scores = output[0]
    # if top label is LABEL_1, use its score. if LABEL_0 won, humor score is 1 - that score
    if scores["label"] == "LABEL_1":
      return scores["score"]
    else:
      return 1 - scores["score"]

  ranked = sorted(texts, key=score, reverse=True)
  for text in texts:
    print(f"{score(text):.3f}  {text}")

In [4]:
import re

def normalize_lyric(text):
  text = text.lower()
  text = re.sub(r"[^\w\s]", "", text) # get rid of punctuation
  text = re.sub(r"\s+", " ", text).strip()
  return text